# CIFAR-10 ResNet V3: Pre-activation ResNet architecture

This notebook trains one custom ResNet **from scratch** on the competition-provided CIFAR-10 images, with a maximum of 5 million trainable parameters.

**Training changes:** Same training approach as V2 but uses a different network of nine pre-activation residual blocks.

1. Start JupyterLab from the repository root.
2. Download the original Kaggle dataset and arrange it as shown in `data/README.md`.
3. Run the cells in order. If the kernel restarts, rerun them; the training function resumes from the last completed epoch.

**Important:** Kaggle test labels are hidden, so the notebook can print validation accuracy, but *test* accuracy must be obtained from your Kaggle submission.

In [ ]:
from pathlib import Path
import sys
import torch
import numpy as np
project = Path.cwd()
if project.name == "notebooks":
    project = project.parent
sys.path.insert(0, str(project))

from src.dataset import get_loaders
from src.models import make_model, count_parameters
from src.training import train, evaluate, plot_training
from src.submission import make_submission

version = "v3"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Data loading and model definition
The data is split into 45,000 training images and 5,000 validation images using a fixed stratified split (`random_state=42`).

In [ ]:
loaders = get_loaders(project, version)
model = make_model(version).to(device)
parameters = count_parameters(model)
print("Training images:", len(loaders["train"].dataset))
print("Validation images:", len(loaders["val"].dataset))
print("Test images:", len(loaders["test"].dataset))
print(f"Trainable parameters: {parameters:,}")
assert parameters <= 5_000_000

## Training
Checkpoints are automatically saved after every epoch. The best weights are determined by **validation accuracy**, not Kaggle test accuracy.

In [ ]:
train_losses, val_accuracies, best_accuracy = train(
    model, loaders, project, version, device
)
print(f"Best validation accuracy: {best_accuracy:.2f}%")
plot_training(train_losses, val_accuracies,
              project / "figures" / f"training_{version}.png")

## Reload best model and check validation accuracy
The training function may end with weights from a later epoch, so load the best checkpoint before predicting.

In [ ]:
model.load_state_dict(torch.load(
    project / "checkpoints" / f"best_resnet_{version}.pth",
    map_location=device, weights_only=True
))
val_acc = evaluate(model, loaders["val"], device)
val_tta = evaluate(model, loaders["val"], device, tta=True)
print(f"Best model validation accuracy: {val_acc:.2f}%")
print(f"Best model validation accuracy (flip TTA): {val_tta:.2f}%")
print(f"Trainable parameters: {count_parameters(model):,}")

## Generate Kaggle submission
This creates predictions for the held-out 2,000 test images, with no need for the test labels. Upload the CSV to Kaggle to obtain the final public accuracy.

In [ ]:
csv_path = make_submission(
    model, loaders["test"], loaders["test_df"],
    project, version, device, tta=True
)
print("Ready to upload:", csv_path)